In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")

df = pd.read_csv(
    RAW / "epa_2025.csv",
    sep=";",               # в немецких CSV разделитель — точка с запятой
    decimal=",",           # десятичная запятая: 2,2 → 2.2
    encoding="utf-8-sig",  # правильно читает умлауты (ä, ö, ü)
    na_values=["X", " "],  # X и пустые клетки = нет данных
    dtype={"Beruf.BA_SCHL": str},  # код профессии — текст, а не число
)
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]  # убираем пустой последний столбец
df.shape

(14491, 21)

In [2]:
df.head()

,Jahr.DESC,Beruf.BA_SCHL,Beruf.DESC,Anforderungsniveau.BA_SCHL,Anforderungsniveau.DESC,Bundesland RZ.BA_SCHL,Bundesland RZ.DESC,Gesamtbewertung,Klassierung Vakanzzeit,Vakanzzeit,...,Arbeitsuchenden-Stellen-Relation,Klassierung Berufsspez. Arbeitslosenquote,Berufsspez. Arbeitslosenquote,Klassierung Veränd. Anteil sv-pfl. beschäftigter Ausländer,Veränd. Anteil sv-pfl. beschäftigter Ausländer,Klassierung Abgangsrate aus Arbeitslosigkeit,Abgangsrate aus Arbeitslosigkeit,Klassierung Entwicklung mittlere Entgelte,Entwicklung mittlere Entgelte,Einstufung
0,2019,1110,Berufe in der Landwirtschaft (o.S.),2,Fachkräfte,d,Deutschland,2.2,2.0,73.0,...,3.5,3.0,0.5,1.0,1.2,3.0,19.8,3.0,10.5,Engpassberuf
1,2019,1111,Berufe in der Landtechnik,2,Fachkräfte,d,Deutschland,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,keine Bewertung möglich
2,2019,1112,Landwirtschaftliche Sachverständige,2,Fachkräfte,d,Deutschland,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,keine Bewertung möglich
3,2019,1113,Berufe landwirtsch.-techn. Laboratorium,2,Fachkräfte,d,Deutschland,1.6,1.0,42.0,...,NaN,2.0,3.2,1.0,0.4,2.0,10.2,2.0,7.5,unter Beobachtung
4,2019,1118,Berufe in der Landwirtschaft (s.s.T.),2,Fachkräfte,d,Deutschland,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,keine Bewertung möglich


In [3]:
de = df[df["Bundesland RZ.BA_SCHL"] == "d"]   # только Германия целиком
de.groupby("Jahr.DESC")["Einstufung"].value_counts().unstack()

Einstufung,Engpassberuf,kein Engpassberuf,keine Bewertung möglich,unter Beobachtung
Jahr.DESC,,,,
2019,178.0,174.0,1558.0,166.0
2020,126.0,230.0,1582.0,138.0
2021,146.0,243.0,1569.0,121.0
2022,202.0,154.0,1566.0,157.0
2023,186.0,168.0,1554.0,171.0
2024,163.0,175.0,NaN,180.0
2025,157.0,201.0,NaN,161.0


In [5]:
rename = {
    "Jahr.DESC": "year",
    "Beruf.BA_SCHL": "kldb",
    "Beruf.DESC": "occupation",
    "Anforderungsniveau.BA_SCHL": "level_code",
    "Anforderungsniveau.DESC": "level",
    "Bundesland RZ.BA_SCHL": "region_code",
    "Bundesland RZ.DESC": "region",
    "Gesamtbewertung": "score",
    "Vakanzzeit": "vacancy_days",
    "Arbeitsuchenden-Stellen-Relation": "seekers_per_vacancy",
    "Berufsspez. Arbeitslosenquote": "unemployment_rate",
    "Veränd. Anteil sv-pfl. beschäftigter Ausländer": "foreign_share_change",
    "Abgangsrate aus Arbeitslosigkeit": "exit_rate",
    "Entwicklung mittlere Entgelte": "wage_growth",
    "Einstufung": "status",
}
status_en = {
    "Engpassberuf": "shortage",
    "unter Beobachtung": "watch",
    "kein Engpassberuf": "no shortage",
    "keine Bewertung möglich": "not rated",
}
clean = df.rename(columns=rename)
clean = clean[list(rename.values())]
clean["status"] = clean["status"].map(status_en)
clean.head()

,year,kldb,occupation,level_code,level,region_code,region,score,vacancy_days,seekers_per_vacancy,unemployment_rate,foreign_share_change,exit_rate,wage_growth,status
0,2019,1110,Berufe in der Landwirtschaft (o.S.),2,Fachkräfte,d,Deutschland,2.2,73.0,3.5,0.5,1.2,19.8,10.5,shortage
1,2019,1111,Berufe in der Landtechnik,2,Fachkräfte,d,Deutschland,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not rated
2,2019,1112,Landwirtschaftliche Sachverständige,2,Fachkräfte,d,Deutschland,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not rated
3,2019,1113,Berufe landwirtsch.-techn. Laboratorium,2,Fachkräfte,d,Deutschland,1.6,42.0,NaN,3.2,0.4,10.2,7.5,watch
4,2019,1118,Berufe in der Landwirtschaft (s.s.T.),2,Fachkräfte,d,Deutschland,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not rated


In [6]:
CLEAN.mkdir(exist_ok=True)
clean.to_csv(CLEAN / "engpass_2019_2025.csv", index=False)
print("Сохранено:", len(clean), "строк")

Сохранено: 14491 строк


In [7]:
import plotly.express as px

de = clean[clean["region_code"] == "d"]
trend = (de[de["status"] == "shortage"]
         .groupby(["year", "level"]).size()
         .reset_index(name="occupations"))

fig = px.line(trend, x="year", y="occupations", color="level", markers=True,
              title="Number of shortage occupations in Germany, by skill level")
fig.show()

## Finding 0: Shortages follow the business cycle

- Shortage occupations: 178 (2019) → 126 (2020, Covid) → peak 202 (2022) → 157 (2025).
- Most shortages are at Fachkraft level (84 of 157 in 2025).
- Shortages among Experten fell fastest since 2022 (40 → 25, −38%); e.g. software development is no longer a shortage occupation.

Source: BA Engpassanalyse 2019–2025.